# Lesson 5.4 - RAG, Tools, and AI Agents (toy pipeline)

This notebook builds a toy but complete RAG + tools + simple agent loop.

## Objectives

- Build a small local corpus and chunk it.
- Run retrieval with TF-IDF vectors and cosine similarity.
- Generate grounded answers from retrieved chunks.
- Add simple tool calls (calculator + search stub) in an agent loop.

## Educational Scope and Production Extension Note
This notebook intentionally uses lightweight simulation/stub components for teaching. Treat outputs as instructional, not production-ready.

For production: replace simulated components with real services/models, add reliability controls, and instrument full observability.

## Setup & Imports

In [1]:
from __future__ import annotations

import re
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Section A - Build Toy Corpus and Chunking

In [2]:
raw_docs = {
    "policy_1": (
        "The incident response policy requires acknowledging critical incidents within 15 minutes, "
        "assigning an incident commander, and publishing status updates every 30 minutes. "
        "After resolution, teams must run a post-incident review within 5 business days."
    ),
    "policy_2": (
        "Customer data retention is 24 months by default unless legal hold applies. "
        "Personally identifiable information must be masked in analytics dashboards. "
        "Access to production data requires approved role-based permissions."
    ),
    "playbook_1": (
        "For model quality regressions, first compare retrieval recall@k and grounding metrics, "
        "then inspect prompt templates and context truncation. Roll back to last stable prompt "
        "version if hallucination rate exceeds threshold."
    ),
    "playbook_2": (
        "On-call escalation matrix: Severity 1 pages immediately to primary and secondary. "
        "Severity 2 pages to primary only. If no acknowledgment in 10 minutes, escalate to manager."
    ),
}


def chunk_text(text: str, chunk_size: int = 45, overlap: int = 10) -> list[str]:
    words = re.findall(r"\S+", text.strip())
    if not words:
        return []
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = words[i : i + chunk_size]
        if chunk:
            chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
    return chunks


records = []
for doc_id, text in raw_docs.items():
    for idx, chunk in enumerate(chunk_text(text, chunk_size=35, overlap=8)):
        records.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_{idx}", "text": chunk})

chunks_df = pd.DataFrame(records)
chunks_df.head(10)

,doc_id,chunk_id,text
0,policy_1,policy_1_0,The incident response policy requires acknowle...
1,policy_2,policy_2_0,Customer data retention is 24 months by defaul...
2,playbook_1,playbook_1_0,"For model quality regressions, first compare r..."
3,playbook_2,playbook_2_0,On-call escalation matrix: Severity 1 pages im...


## Section B - Embedding + Retrieval (TF-IDF)

In [3]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
chunk_matrix = vectorizer.fit_transform(chunks_df["text"])


def retrieve(query: str, top_k: int = 3) -> pd.DataFrame:
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, chunk_matrix).ravel()
    idx = np.argsort(-sims)[:top_k]
    out = chunks_df.iloc[idx].copy()
    out["score"] = sims[idx]
    return out[["doc_id", "chunk_id", "score", "text"]].reset_index(drop=True)


retrieve("What should we do when hallucination rate increases?")

,doc_id,chunk_id,score,text
0,playbook_1,playbook_1_0,0.224246,"For model quality regressions, first compare r..."
1,policy_1,policy_1_0,0.000000,The incident response policy requires acknowle...
2,policy_2,policy_2_0,0.000000,Customer data retention is 24 months by defaul...


## Section C - Retrieval-Augmented Answer Generation

In [4]:
def build_context(query: str, top_k: int = 3) -> str:
    hits = retrieve(query, top_k=top_k)
    context_lines = []
    for i, row in hits.iterrows():
        context_lines.append(f"[{i+1}] ({row['chunk_id']}) {row['text']}")
    return "\n".join(context_lines)


def rag_answer(query: str, top_k: int = 3) -> str:
    hits = retrieve(query, top_k=top_k)
    if hits.empty:
        return "I do not have enough retrieved evidence to answer."

    top = hits.iloc[0]
    answer = (
        f"Based on retrieved guidance, the best immediate action is to follow: {top['text']} "
        f"(source: {top['chunk_id']})."
    )
    return answer


query = "How should we respond to model hallucination regressions?"
print("Context:\n", build_context(query, top_k=3))
print("\nAnswer:\n", rag_answer(query, top_k=3))

Context:
 [1] (playbook_1_0) For model quality regressions, first compare retrieval recall@k and grounding metrics, then inspect prompt templates and context truncation. Roll back to last stable prompt version if hallucination rate exceeds threshold.
[2] (playbook_2_0) On-call escalation matrix: Severity 1 pages immediately to primary and secondary. Severity 2 pages to primary only. If no acknowledgment in 10 minutes, escalate to manager.
[3] (policy_2_0) Customer data retention is 24 months by default unless legal hold applies. Personally identifiable information must be masked in analytics dashboards. Access to production data requires approved role-based permissions.

Answer:
 Based on retrieved guidance, the best immediate action is to follow: For model quality regressions, first compare retrieval recall@k and grounding metrics, then inspect prompt templates and context truncation. Roll back to last stable prompt version if hallucination rate exceeds threshold. (source: playbook_1_

## Section D - Tool Use and Simple Agent Loop

In [5]:
def tool_calculator(expression: str) -> str:
    if not re.fullmatch(r"[0-9\s\+\-\*\/\(\)\.]+", expression):
        return "calculator_error: invalid expression"
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"calculator_result: {result}"
    except Exception as exc:
        return f"calculator_error: {exc}"


def tool_time(_: str = "") -> str:
    return f"current_utc_time: {datetime.utcnow().isoformat()}Z"


def tool_doc_search(query: str) -> str:
    hits = retrieve(query, top_k=2)
    if hits.empty:
        return "search_result: no relevant chunks"
    lines = [f"{row.chunk_id} ({row.score:.3f}): {row.text}" for row in hits.itertuples(index=False)]
    return "search_result:\n" + "\n".join(lines)


def simple_agent(user_query: str) -> str:
    q = user_query.lower().strip()

    if re.search(r"\d+[\d\s\+\-\*\/\(\)\.]*", q) and any(op in q for op in ["+", "-", "*", "/"]):
        expr = re.findall(r"[0-9\s\+\-\*\/\(\)\.]+", q)
        expr = max(expr, key=len).strip() if expr else ""
        return tool_calculator(expr)

    if "time" in q or "date" in q:
        return tool_time()

    return rag_answer(user_query, top_k=3)


queries = [
    "What is the acknowledgment SLA for critical incidents?",
    "What should we do when hallucination rate is high?",
    "What is 25 * (4 + 6)?",
    "What time is it now?",
]

for q in queries:
    print("Q:", q)
    print("A:", simple_agent(q))
    print("-" * 80)

Q: What is the acknowledgment SLA for critical incidents?
A: Based on retrieved guidance, the best immediate action is to follow: The incident response policy requires acknowledging critical incidents within 15 minutes, assigning an incident commander, and publishing status updates every 30 minutes. After resolution, teams must run a post-incident review within 5 business days. (source: policy_1_0).
--------------------------------------------------------------------------------
Q: What should we do when hallucination rate is high?
A: Based on retrieved guidance, the best immediate action is to follow: For model quality regressions, first compare retrieval recall@k and grounding metrics, then inspect prompt templates and context truncation. Roll back to last stable prompt version if hallucination rate exceeds threshold. (source: playbook_1_0).
--------------------------------------------------------------------------------
Q: What is 25 * (4 + 6)?
A: calculator_result: 250
------------

/tmp/ipykernel_576122/1313397349.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return f"current_utc_time: {datetime.utcnow().isoformat()}Z"


## Connect to Theory

- Chunking and retrieval quality determine what evidence reaches the generator.
- The toy RAG answer uses retrieved text with source IDs as simple grounding.
- Tool-calling augments LLM-like behavior with deterministic capabilities (math/time/search).
- Agent routing logic should be observable and testable to prevent silent tool misuse.

## Production Extension Checklist
- Replace simulated/stub components with production services or managed endpoints.
- Add structured logging, tracing IDs, and latency/error dashboards.
- Add input/output validation and policy/safety guardrails.
- Add retry/timeouts, rate limiting, and fallback behavior.
- Add offline/online evaluation gates before release.
- Add secrets management and environment-specific configuration.
- Add CI checks and smoke tests for critical paths.

## Business Case Studies & Exceptions

### Case 1: Enterprise Knowledge Assistant

- **Goal**: answer policy questions with citations.
- **Pattern**: retrieval + reranking + prompt grounding + source-linked output.
- **Exception**: stale indexing can return outdated policy data.

### Case 2: Operations Agent with Guarded Tools

- **Goal**: automate triage workflows (status checks, metric queries, ticket drafting).
- **Pattern**: intent routing + schema-validated tool calls + audit logs.
- **Exception**: unrestricted tools can produce unsafe side effects; enforce role-based permissions.

### Case 3: RAG Ops / Observability

Track at least:

- retrieval recall@k,
- groundedness/citation accuracy,
- tool-call error rate,
- end-to-end latency split by stage.

## Interview Questions & Answers

1. **Q: What is RAG?**
   **A:** A pattern that retrieves external context and augments prompts so generation is grounded in evidence.

2. **Q: What are the main components of a RAG pipeline?**
   **A:** Ingestion/chunking, embeddings/indexing, retrieval, optional reranking, prompt augmentation, generation.

3. **Q: RAG vs fine-tuning?**
   **A:** RAG updates knowledge by re-indexing documents; fine-tuning updates model behavior via parameter changes.

4. **Q: Why is chunking important?**
   **A:** It controls retrieval granularity and context relevance.

5. **Q: What is a reranker?**
   **A:** A second-stage model that reorders retrieved candidates to improve precision.

6. **Q: What are common RAG failure modes?**
   **A:** Missing relevant docs, poor chunk boundaries, stale data, and hallucinated citations.

7. **Q: How do tools improve LLM systems?**
   **A:** They provide deterministic access to external computation and data sources.

8. **Q: What is an AI agent in this context?**
   **A:** A system that plans and executes multi-step actions, often with tools and state management.

9. **Q: How do you reduce dangerous tool use?**
   **A:** Enforce schemas, permissions, allowlists, and audit logging.

10. **Q: How do you evaluate RAG quality?**
   **A:** Measure retrieval metrics, grounded answer quality, user outcomes, latency, and cost.

11. **Q: Why is observability critical for agents?**
   **A:** Multi-step flows fail in non-obvious ways; traces and logs are required for debugging and trust.

12. **Q: What is a practical rollout strategy?**
   **A:** Start read-only with citations, validate quality, then gradually enable higher-impact tools.
